In [ ]:
from typing import TypedDict, Any, Annotated
import operator


class DataDescriptionState(TypedDict, total=False):
    # Input from main orchestrator
    dataset_path: str
    target_column: str | None

    # Optional config
    artifacts_dir: str

    # Stage results
    basic_overview: dict[str, Any]
    domain_understanding: dict[str, Any]
    schema_detection: dict[str, Any]
    data_quality: dict[str, Any]
    statistics: dict[str, Any]

    # Final compact result of Data Description Agent
    final_result: dict[str, Any]

    # Output for the next agent
    next_input: dict[str, Any]

    # Key detected task info
    task_type: str | None
    schema: dict[str, Any]
    data_summary: dict[str, Any]

    # Artifacts and logs
    artifacts: dict[str, str]
    logs: Annotated[list[dict[str, Any]], operator.add]

    # Status
    status: str
    error: str | None

In [ ]:
DATA_DESCRIPTION_ROLE_PROMPT = """
You are a deterministic data analysis agent.

Your role:
- You strictly follow instructions.
- You write correct and minimal Python code when using python_interpreter_tool.
- You return ONLY valid JSON when JSON is requested.
- You never explain anything when JSON is requested.
- You never add markdown, comments, debug text, or extra text when JSON is requested.
- You never hallucinate columns, values, metrics, or conclusions.
- You rely only on the actual dataset, computed results, and previous stage outputs.
- If something is unclear, make the safest conservative assumption and record it in notes if the schema allows it.

Rules:
- Output must be valid compact JSON only.
- No extra text before or after JSON.
- No markdown.
- No explanations.
""".strip()


def run_stage_with_llm(stage_name, task):
    print(f"\n========== START STAGE: {stage_name} ==========")

    try:
        result = data_description_stage_agent.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": DATA_DESCRIPTION_ROLE_PROMPT + "\n\n" + task,
                }
            ]
        })

        raw = result["messages"][-1].content.strip()

        print("\nRAW RESPONSE:")
        print(raw)

        parsed = extract_json(raw)

        print(f"========== END STAGE: {stage_name} | status=success ==========")
        return parsed

    except Exception as e:
        print(f"========== END STAGE: {stage_name} | status=failed ==========")
        print(f"ERROR: {e}")
        raise


In [ ]:
from pathlib import Path
import json


def dd_basic_overview_node(state):
    dataset_path = state["dataset_path"]

    task = f"""
Stage: basic_overview

Dataset path:
{dataset_path}

Goal:
Create a reliable first overview of the raw dataset. Use only values computed from the dataset.

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Compute directly with pandas:
   - n_rows: number of rows
   - n_columns: number of columns
   - columns: list of column names in original order
   - dtypes: pandas dtype for each column as string
   - missing_values_total: total number of missing values in the whole dataset
   - duplicate_rows: number of fully duplicated rows
   - memory_usage_mb: total dataframe memory usage in megabytes, rounded to 4 decimals
3. Do not preprocess data.
4. Do not modify df.
5. Do not print Python code, markdown, comments, explanations, or debug output.
6. Do not return the full dataframe.
7. Do not keep the full dataframe in memory for next stages.
8. Each later stage will read the dataset again if needed.

Validation rules:
- All numeric values must be real computed values, not estimates.
- columns length must equal n_columns.
- dtypes keys must exactly match columns.
- Return only compact valid JSON.

Return JSON only:
{{
  "stage": "basic_overview",
  "status": "success",
  "result": {{
    "n_rows": 0,
    "n_columns": 0,
    "columns": [],
    "dtypes": {{}},
    "missing_values_total": 0,
    "duplicate_rows": 0,
    "memory_usage_mb": 0.0
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("basic_overview", task)
        basic_overview = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        basic_overview_path = artifacts_dir / "basic_overview.json"

        with open(basic_overview_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "data_description_agent",
            "stage": "basic_overview",
            "status": "success",
            "summary": "Basic dataset overview completed.",
            "artifacts": {
                "basic_overview_path": str(basic_overview_path),
            },
        }

        return {
            "basic_overview": basic_overview,
            "artifacts": {
                **state.get("artifacts", {}),
                "basic_overview_path": str(basic_overview_path),
            },
            "logs": [new_log],
            "status": "running",
            "error": None,
        }

    except Exception as e:
        new_log = {
            "agent": "data_description_agent",
            "stage": "basic_overview",
            "status": "failed",
            "summary": "Basic overview failed.",
            "reason": str(e),
        }

        return {
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }

In [ ]:
from pathlib import Path
import json


def dd_domain_understanding_node(state):
    dataset_path = state["dataset_path"]
    basic = state.get("basic_overview", {})

    task = f"""
Stage: domain_understanding

Dataset path:
{dataset_path}

Previous result:
{json.dumps(basic, ensure_ascii=False, indent=2)}

Goal:
Infer the dataset domain and row meaning from column names and small samples only.

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Inspect only compact evidence:
   - df.head(3)
   - column names
   - dtypes
   - up to 3 non-null example values from important object/string columns
3. Do not print Python code, markdown, comments, explanations, or debug output.
4. Do not print the full dataframe.
5. Do not keep the full dataframe in memory for next stages.

Infer and return:
- domain_summary: one short sentence describing the likely subject area
- row_meaning: one short sentence describing what one row likely represents
- column_groups_description: object where keys are semantic groups and values are lists of column names
- confidence: "low", "medium", or "high"
- notes: short list of conservative caveats if inference is uncertain

Important rules:
- Do not preprocess data.
- Do not modify df.
- Base the inference only on column names and compact sample values.
- If the domain is not clear, say so in notes and use confidence="low".
- Return only compact valid JSON.

Return JSON only:
{{
  "stage": "domain_understanding",
  "status": "success",
  "result": {{
    "domain_summary": "",
    "row_meaning": "",
    "column_groups_description": {{}},
    "confidence": "low",
    "notes": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("domain_understanding", task)
        domain_understanding = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        domain_understanding_path = artifacts_dir / "domain_understanding.json"

        with open(domain_understanding_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "data_description_agent",
            "stage": "domain_understanding",
            "status": "success",
            "summary": "Domain understanding completed.",
            "artifacts": {
                "domain_understanding_path": str(domain_understanding_path),
            },
        }

        return {
            "domain_understanding": domain_understanding,
            "artifacts": {
                **state.get("artifacts", {}),
                "domain_understanding_path": str(domain_understanding_path),
            },
            "logs": [new_log],
            "status": "running",
            "error": None,
        }

    except Exception as e:
        new_log = {
            "agent": "data_description_agent",
            "stage": "domain_understanding",
            "status": "failed",
            "summary": "Domain understanding failed.",
            "reason": str(e),
        }

        return {
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }

In [ ]:
from pathlib import Path
import json


def dd_schema_detection_node(state):
    dataset_path = state["dataset_path"]
    target_column = state.get("target_column")
    basic = state.get("basic_overview", {})
    domain = state.get("domain_understanding", {})

    task = f"""
Stage: schema_detection

Dataset path:
{dataset_path}

Target column from user:
{target_column}

Previous results:
basic_overview:
{json.dumps(basic, ensure_ascii=False, indent=2)}

domain_understanding:
{json.dumps(domain, ensure_ascii=False, indent=2)}

Goal:
Detect the schema of the raw dataset and identify the target column and task type when possible.

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. For every column compute:
   - pandas dtype
   - number of unique values including non-null values only
   - number of missing values
   - missing ratio
   - for object/string columns: average string length, max string length, and up to 3 non-null sample values
3. Detect schema groups:
   - numeric
   - categorical
   - text
   - datetime
   - boolean_like
   - id
   - target
4. Determine task_type:
   - "regression" if target is numeric continuous
   - "binary_classification" if target has exactly 2 distinct non-null values
   - "multiclass_classification" if target has more than 2 discrete classes
   - "unknown" if target is missing or unclear

Schema classification rules:
- A column must appear in at most one feature group, except target tracking.
- target must contain only the selected target column if it exists.
- If target_column does not exist in df, set target_column to null and task_type to "unknown".
- If target_column is None, infer target only if obvious from column names such as target, label, y, price, outcome, class. Otherwise use null.
- ID-like columns such as id, *_id, user_id, host_id, listing_id must be id, not numeric.
- Numeric columns are int/float columns that are not id-like, boolean-like, or target.
- Datetime detection applies only to object/string/datetime columns. Do NOT classify numeric columns as datetime.
- Boolean-like columns have only boolean-like values such as true/false, yes/no, 0/1.
- Categorical columns are short string/object columns with repeated values and low or moderate cardinality.
- Text columns are long free-form strings, descriptions, natural language, lists, or high-cardinality strings.
- Columns like name, description, overview, about, amenities, comments, review, summary are text when they contain phrases or lists.

Important rules:
- Do not preprocess data.
- Do not modify df.
- Do not print Python code, markdown, comments, explanations, or debug output.
- Do not return the full dataframe.
- Do not keep the full dataframe in memory for next stages.
- Return only compact valid JSON.

Return JSON only:
{{
  "stage": "schema_detection",
  "status": "success",
  "result": {{
    "target_column": null,
    "task_type": "unknown",
    "schema": {{
      "numeric": [],
      "categorical": [],
      "text": [],
      "datetime": [],
      "boolean_like": [],
      "id": [],
      "target": []
    }},
    "column_profile": {{}},
    "schema_notes": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("schema_detection", task)
        schema_detection = parsed.get("result", {})

        detected_target_column = schema_detection.get("target_column")
        detected_task_type = schema_detection.get("task_type")
        detected_schema = schema_detection.get("schema", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        schema_detection_path = artifacts_dir / "schema_detection.json"

        with open(schema_detection_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "data_description_agent",
            "stage": "schema_detection",
            "status": "success",
            "summary": "Schema detection completed.",
            "artifacts": {
                "schema_detection_path": str(schema_detection_path),
            },
        }

        return {
            "schema_detection": schema_detection,

            # полезно сохранить наверх внутри DataDescriptionState
            "target_column": detected_target_column,
            "task_type": detected_task_type,
            "schema": detected_schema,

            "artifacts": {
                **state.get("artifacts", {}),
                "schema_detection_path": str(schema_detection_path),
            },
            "logs": [new_log],
            "status": "running",
            "error": None,
        }

    except Exception as e:
        new_log = {
            "agent": "data_description_agent",
            "stage": "schema_detection",
            "status": "failed",
            "summary": "Schema detection failed.",
            "reason": str(e),
        }

        return {
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }

In [ ]:
from pathlib import Path
import json


def dd_data_quality_node(state):
    dataset_path = state["dataset_path"]
    schema_detection = state.get("schema_detection", {})

    task = f"""
Stage: data_quality

Dataset path:
{dataset_path}

Schema result:
{json.dumps(schema_detection, ensure_ascii=False, indent=2)}

Goal:
Measure data quality problems in the raw dataset using computed values only.

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Use schema["numeric"] from the provided schema result.
3. Compute:
   - missing_values: missing values by column
   - missing_values_total: total missing values in the dataset
   - missing_ratio_by_column: missing ratio by column rounded to 6 decimals
   - duplicate_rows: number of fully duplicated rows
   - constant_columns: columns with exactly 1 unique non-null value
   - high_cardinality_columns: columns where unique non-null values are more than 90% of rows
   - high_missing_columns: columns where missing ratio is greater than 0.5
   - numeric_outliers_iqr: outlier counts for numeric schema columns using IQR rule only
4. IQR outlier rule:
   - Q1 = 25th percentile
   - Q3 = 75th percentile
   - IQR = Q3 - Q1
   - lower = Q1 - 1.5 * IQR
   - upper = Q3 + 1.5 * IQR
   - outliers are values lower than lower or greater than upper

Important rules:
- Do not preprocess data.
- Do not modify df.
- Do not print Python code, markdown, comments, explanations, or debug output.
- Do not return the full dataframe.
- Do not keep the full dataframe in memory for next stages.
- If schema["numeric"] is empty, return an empty numeric_outliers_iqr dictionary.
- The values in final JSON must be copied exactly from variables calculated by python_interpreter_tool.
- Do not estimate or invent values.
- Return only compact valid JSON.

Return JSON only:
{{
  "stage": "data_quality",
  "status": "success",
  "result": {{
    "missing_values": {{}},
    "missing_values_total": 0,
    "missing_ratio_by_column": {{}},
    "duplicate_rows": 0,
    "constant_columns": [],
    "high_cardinality_columns": [],
    "high_missing_columns": [],
    "numeric_outliers_iqr": {{}}
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("data_quality", task)
        data_quality = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        data_quality_path = artifacts_dir / "data_quality.json"

        with open(data_quality_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "data_description_agent",
            "stage": "data_quality",
            "status": "success",
            "summary": "Data quality analysis completed.",
            "artifacts": {
                "data_quality_path": str(data_quality_path),
            },
        }

        return {
            "data_quality": data_quality,
            "artifacts": {
                **state.get("artifacts", {}),
                "data_quality_path": str(data_quality_path),
            },
            "logs": [new_log],
            "status": "running",
            "error": None,
        }

    except Exception as e:
        new_log = {
            "agent": "data_description_agent",
            "stage": "data_quality",
            "status": "failed",
            "summary": "Data quality analysis failed.",
            "reason": str(e),
        }

        return {
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }

In [ ]:
from pathlib import Path
import json


def dd_statistics_node(state):
    dataset_path = state["dataset_path"]
    schema_detection = state.get("schema_detection", {})

    task = f"""
Stage: statistics

Dataset path:
{dataset_path}

Schema result:
{json.dumps(schema_detection, ensure_ascii=False, indent=2)}

Goal:
Calculate compact statistical metadata for the detected schema and target column.

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Use schema and target_column from the provided schema result.
3. For numeric columns compute compact metadata:
   - count, mean, std, min, median, max for each numeric column
4. For categorical columns compute compact metadata:
   - unique count
   - top value
   - top value frequency
5. For text columns compute compact metadata:
   - non-null count
   - average string length
   - max string length
6. For datetime columns compute compact metadata:
   - min date
   - max date
   - non-null count
7. For target_column if it exists compute:
   - target_type
   - missing count
   - unique count
   - distribution for classification target
   - summary statistics for regression target
8. Save detailed statistics into the stage JSON artifact by returning them inside result.

Important rules:
- Do not preprocess data.
- Do not modify df.
- Do not print Python code, markdown, comments, explanations, or debug output.
- Do not return the full dataframe.
- Do not keep the full dataframe in memory for next stages.
- Round floats to 6 decimals.
- Use null for values that cannot be computed.
- Return only compact valid JSON.

Return JSON only:
{{
  "stage": "statistics",
  "status": "success",
  "result": {{
    "numeric_columns_count": 0,
    "categorical_columns_count": 0,
    "text_columns_count": 0,
    "datetime_columns_count": 0,
    "target_column": null,
    "numeric_summary": {{}},
    "categorical_summary": {{}},
    "text_summary": {{}},
    "datetime_summary": {{}},
    "target_summary": {{}},
    "statistics_note": "Detailed statistics were calculated from the raw dataset."
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("statistics", task)
        statistics = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        statistics_path = artifacts_dir / "statistics.json"

        with open(statistics_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        statistics = {
            **statistics,
            "statistics_path": str(statistics_path),
        }

        new_log = {
            "agent": "data_description_agent",
            "stage": "statistics",
            "status": "success",
            "summary": "Statistics stage completed.",
            "artifacts": {
                "statistics_path": str(statistics_path),
            },
        }

        return {
            "statistics": statistics,
            "artifacts": {
                **state.get("artifacts", {}),
                "statistics_path": str(statistics_path),
            },
            "logs": [new_log],
            "status": "running",
            "error": None,
        }

    except Exception as e:
        new_log = {
            "agent": "data_description_agent",
            "stage": "statistics",
            "status": "failed",
            "summary": "Statistics calculation failed.",
            "reason": str(e),
        }

        return {
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }

In [ ]:
from pathlib import Path
import json


def dd_build_final_result_node(state):
    if state.get("error"):
        return {
            "logs": [{
                "agent": "data_description_agent",
                "stage": "build_final_result",
                "status": "skipped",
                "summary": "Final result build was skipped because previous stage failed.",
                "reason": state.get("error"),
            }],
        }

    basic = state.get("basic_overview", {})
    domain = state.get("domain_understanding", {})
    schema_detection = state.get("schema_detection", {})
    data_quality = state.get("data_quality", {})
    statistics = state.get("statistics", {})
    previous_artifacts = state.get("artifacts", {})

    required_basic_keys = ["n_rows", "n_columns", "columns", "dtypes"]
    missing_basic_keys = [key for key in required_basic_keys if key not in basic]

    if missing_basic_keys:
        error = f"Cannot build final result. Missing basic_overview keys: {missing_basic_keys}"
        return {
            "error": error,
            "logs": [{
                "agent": "data_description_agent",
                "stage": "build_final_result",
                "status": "failed",
                "summary": "Final result build failed.",
                "reason": error,
            }],
        }

    schema = schema_detection.get("schema", {})

    for key in ["numeric", "categorical", "text", "datetime"]:
        schema.setdefault(key, [])

    for key in ["boolean_like", "id", "target"]:
        schema.setdefault(key, [])

    target_column = schema_detection.get("target_column")
    task_type = schema_detection.get("task_type", "unknown")

    if target_column is None:
        task_type = "unknown"

    artifacts_dir = Path("artifacts/data_description")
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    summary_path = artifacts_dir / "data_description_summary.json"
    eda_artifacts_path = artifacts_dir / "eda_artifacts.json"
    report_path = artifacts_dir / "data_description_report.md"

    statistics_path = statistics.get(
        "statistics_path",
        previous_artifacts.get("statistics_path"),
    )

    artifacts = {
        **previous_artifacts,
        "data_description_summary_path": str(summary_path),
        "eda_artifacts_path": str(eda_artifacts_path),
        "data_description_report_path": str(report_path),
        "statistics_path": statistics_path,
        "eda_stages_dir": "artifacts/data_description/stages",
    }

    data_summary = {
        "n_rows": basic.get("n_rows"),
        "n_columns": basic.get("n_columns"),
        "domain_summary": domain.get("domain_summary"),
        "row_meaning": domain.get("row_meaning"),
        "missing_values_total": data_quality.get("missing_values_total"),
        "duplicate_rows": data_quality.get("duplicate_rows"),
        "statistics_path": statistics_path,
        "eda_artifacts_path": str(eda_artifacts_path),
    }

    final_result = {
        "agent": "data_description_agent",
        "status": "success",
        "skipped": False,
        "summary": (
            f"Dataset contains {basic.get('n_rows')} rows and "
            f"{basic.get('n_columns')} columns. Task type: {task_type}."
        ),
        "decisions": {
            "domain_summary": domain.get("domain_summary"),
            "row_meaning": domain.get("row_meaning"),
            "target_column": target_column,
            "task_type": task_type,
            "schema": schema,
            "data_quality_summary": {
                "missing_values_total": data_quality.get("missing_values_total"),
                "duplicate_rows": data_quality.get("duplicate_rows"),
                "constant_columns_count": len(data_quality.get("constant_columns", [])),
                "high_cardinality_columns_count": len(data_quality.get("high_cardinality_columns", [])),
            },
        },
        "artifacts": artifacts,
        "next_input": {
            "target_column": target_column,
            "task_type": task_type,
            "schema": schema,
            "data_summary": data_summary,
        },
        "reason": None,
    }

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(final_result, f, ensure_ascii=False, indent=2)

    eda_artifacts = {
        "basic_overview": basic,
        "domain_understanding": domain,
        "schema_detection": schema_detection,
        "data_quality": data_quality,
        "statistics": statistics,
        "stage_artifact_paths": {
            key: value
            for key, value in artifacts.items()
            if key.endswith("_path") or key.endswith("_dir")
        },
    }

    with open(eda_artifacts_path, "w", encoding="utf-8") as f:
        json.dump(eda_artifacts, f, ensure_ascii=False, indent=2)

    report_md = (
        "# Data Description Report\n\n"
        "## Summary\n"
        f"{final_result.get('summary')}\n\n"
        "## Domain\n"
        f"{domain.get('domain_summary')}\n\n"
        "## Row meaning\n"
        f"{domain.get('row_meaning')}\n\n"
        "## Target\n"
        f"- Target column: {target_column}\n"
        f"- Task type: {task_type}\n\n"
        "## Dataset shape\n"
        f"- Rows: {basic.get('n_rows')}\n"
        f"- Columns: {basic.get('n_columns')}\n\n"
        "## Data quality summary\n"
        f"- Total missing values: {data_quality.get('missing_values_total')}\n"
        f"- Duplicate rows: {data_quality.get('duplicate_rows')}\n"
        f"- Constant columns count: {len(data_quality.get('constant_columns', []))}\n"
        f"- High-cardinality columns count: {len(data_quality.get('high_cardinality_columns', []))}\n\n"
        "## Schema\n"
        "```json\n"
        f"{json.dumps(schema, ensure_ascii=False, indent=2)}\n"
        "```\n\n"
        "## Artifacts\n"
        f"- Summary: {summary_path}\n"
        f"- EDA artifacts: {eda_artifacts_path}\n"
        f"- Statistics: {statistics_path}\n"
        f"- Stage artifacts dir: {artifacts.get('eda_stages_dir')}\n"
    )

    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_md)

    print("\nRAW RESPONSE: build_final_result")
    print(json.dumps(final_result, ensure_ascii=False, indent=2))
    print("END RAW RESPONSE\n")

    return {
        "final_result": final_result,
        "artifacts": artifacts,
        "logs": [{
            "agent": "data_description_agent",
            "stage": "build_final_result",
            "status": "success",
            "summary": "Final compact data description result was built and saved.",
            "artifacts": {
                "data_description_summary_path": str(summary_path),
                "eda_artifacts_path": str(eda_artifacts_path),
                "data_description_report_path": str(report_path),
            },
        }],
    }

In [ ]:
def route_after_dd_stage(state):
    if state.get("error"):
        return "stop"
    return "continue"

In [ ]:
from langgraph.graph import StateGraph, START, END


def build_data_description_graph():
    workflow = StateGraph(DataDescriptionState)

    workflow.add_node("basic_overview", dd_basic_overview_node)
    workflow.add_node("domain_understanding", dd_domain_understanding_node)
    workflow.add_node("schema_detection", dd_schema_detection_node)
    workflow.add_node("data_quality", dd_data_quality_node)
    workflow.add_node("statistics", dd_statistics_node)
    workflow.add_node("build_final_result", dd_build_final_result_node)

    workflow.add_edge(START, "basic_overview")

    workflow.add_conditional_edges(
        "basic_overview",
        route_after_dd_stage,
        {"continue": "domain_understanding", "stop": END},
    )

    workflow.add_conditional_edges(
        "domain_understanding",
        route_after_dd_stage,
        {"continue": "schema_detection", "stop": END},
    )

    workflow.add_conditional_edges(
        "schema_detection",
        route_after_dd_stage,
        {"continue": "data_quality", "stop": END},
    )

    workflow.add_conditional_edges(
        "data_quality",
        route_after_dd_stage,
        {"continue": "statistics", "stop": END},
    )

    workflow.add_conditional_edges(
        "statistics",
        route_after_dd_stage,
        {"continue": "build_final_result", "stop": END},
    )

    workflow.add_conditional_edges(
        "build_final_result",
        route_after_dd_stage,
        {"continue": END, "stop": END},
    )

    return workflow.compile()

In [ ]:
def data_description_agent_node(state):
    dataset_path = state.get("dataset_path")
    target_column = state.get("target_column")

    if not dataset_path:
        log_record = {
            "agent": "data_description_agent",
            "status": "failed",
            "skipped": False,
            "summary": "Dataset path is missing.",
            "decisions": {},
            "artifacts": {},
            "next_input": {},
            "reason": "dataset_path is None",
        }

        return {
            "current_step": "data_description_agent",
            "error": "dataset_path is missing",
            "status": "failed",
            "logs": [log_record],
        }

    try:
        data_description_app = build_data_description_graph()

        dd_state = data_description_app.invoke(
            {
                "dataset_path": dataset_path,
                "target_column": target_column,
                "artifacts_dir": "artifacts/data_description",
                "logs": [],
                "error": None,
                "artifacts": {},
                "status": "running",
                "next_input": {},
            },
            {"recursion_limit": 20},
        )

        final_result = dd_state.get("final_result", {})

        if dd_state.get("error") or not final_result:
            error = dd_state.get("error") or "Data Description subgraph did not produce final_result."

            log_record = {
                "agent": "data_description_agent",
                "status": "failed",
                "skipped": False,
                "summary": "Data Description subgraph failed.",
                "decisions": {},
                "artifacts": dd_state.get("artifacts", {}),
                "next_input": {},
                "reason": error,
                "subgraph_logs": dd_state.get("logs", []),
            }

            return {
                "current_step": "data_description_agent",
                "error": error,
                "status": "failed",
                "artifacts": {
                    **state.get("artifacts", {}),
                    **dd_state.get("artifacts", {}),
                },
                "logs": [log_record],
            }

        next_input = final_result.get("next_input", {})
        decisions = final_result.get("decisions", {})
        artifacts = final_result.get("artifacts", {})

        schema = next_input.get("schema") or decisions.get("schema", {})
        data_summary = next_input.get("data_summary", {})

        detected_target_column = next_input.get("target_column", target_column)
        detected_task_type = next_input.get("task_type", state.get("task_type"))

        log_record = {
            "agent": "data_description_agent",
            "status": final_result.get("status", "success"),
            "skipped": final_result.get("skipped", False),
            "summary": final_result.get("summary", "Data description completed."),
            "decisions": decisions,
            "artifacts": artifacts,
            "next_input": next_input,
            "reason": final_result.get("reason"),
            "subgraph_logs": dd_state.get("logs", []),
        }

        return {
            "current_step": "data_description_agent",

            # Main detected task info
            "target_column": detected_target_column,
            "task_type": detected_task_type,
            "schema": schema,
            "data_summary": data_summary,

            # Important: this is the input for Data Preparation Agent
            "next_input": next_input,

            # Artifacts
            "artifacts": {
                **state.get("artifacts", {}),
                **artifacts,
            },

            # Logs are accumulated by operator.add in AgentState
            "logs": [log_record],

            # Clear previous orchestrator decision after executing the selected agent
            "next_action": None,
            "orchestration_decision": {},

            "status": "running",
            "error": None,
        }

    except Exception as e:
        log_record = {
            "agent": "data_description_agent",
            "status": "failed",
            "skipped": False,
            "summary": "Data Description subgraph failed.",
            "decisions": {},
            "artifacts": {},
            "next_input": {},
            "reason": str(e),
        }

        return {
            "current_step": "data_description_agent",
            "error": str(e),
            "status": "failed",
            "logs": [log_record],
        }